# A Canonical-State Digital Twin Framework for Occupancy-Aware Smart-Room Power Forecasting and Decision Support

**Consolidated reviewer-facing research workflow**

> This consolidated notebook was added after experiment-v1.0 to provide a single reviewer-facing interface. It does not modify the frozen scientific experiment.

This notebook is a read-only presentation layer over the verified artifacts in `results/`. The scientific source of truth remains the Python implementation under `src/`, while notebooks 01–06 remain the frozen granular reproducibility evidence.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import inspect
import json
import os
import re
import subprocess
import sys

REPO_URL = 'https://github.com/rehanalfarizu/new_jurnal.git'
COLAB_REPO_PATH = Path('/content/new_jurnal')
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'results').is_dir() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Repository root containing results/ and src/ was not found.')

if IN_COLAB:
    if not COLAB_REPO_PATH.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(COLAB_REPO_PATH)], check=True)
    ROOT = COLAB_REPO_PATH
    os.chdir(ROOT)
    if not (ROOT / 'results').is_dir() or not (ROOT / 'src').is_dir():
        raise RuntimeError(f'Existing Colab repository path is incomplete: {ROOT}')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')
    ], check=True)
else:
    ROOT = find_repository_root(Path.cwd())
    os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Code, HTML, Image, Markdown, display

RECOMPUTE_EXPERIMENTS = False
SENSOR_DATA_PATH = Path(os.environ.get('SENSOR_DATA_PATH', 'data/raw/sensor_data.csv'))

def read_csv(relative_path: str, **kwargs) -> pd.DataFrame:
    return pd.read_csv(ROOT / relative_path, **kwargs)

def read_json(relative_path: str) -> dict:
    return json.loads((ROOT / relative_path).read_text(encoding='utf-8'))

def show_figure(relative_path: str, width: int = 1050) -> None:
    display(Image(filename=str(ROOT / relative_path), width=width))

def display_readable_table(frame: pd.DataFrame) -> None:
    styles = (
        '<style>table.reviewer-evidence {width:100%;} '
        'table.reviewer-evidence th, table.reviewer-evidence td {'
        'text-align:left; vertical-align:top; white-space:normal; '
        'overflow-wrap:anywhere; max-width:420px;}</style>'
    )
    display(HTML(styles + frame.to_html(index=False, escape=True, classes='reviewer-evidence')))

assert RECOMPUTE_EXPERIMENTS is False
print(f'Repository root: {ROOT}')
print(f'Google Colab detected: {IN_COLAB}')
print('Execution mode: Verified frozen artifacts')
print('Raw dataset required: No')
print('Model retraining: No')
print(f'Optional reproduction dataset location: {SENSOR_DATA_PATH}')

## 1. Research Overview

The study evaluates a Digital Twin research framework using historical IoT telemetry from one Smart Room, one gateway, and one observation period. The workflow covers canonical-state evaluation, 30-minute-ahead power forecasting, occupancy ablation, and transparent rule-based decision support.

**Forecast target: `power(t + 30 min)` in W.** It is not electrical energy in Wh. The experiment does not evaluate realized energy savings or autonomous control, and it does not support a cross-building generalization claim.

## 2. Reproducibility and Freeze Status

The freeze facts below are read from the historical reproducibility manifest. The current repository commit is shown separately because this presentation notebook is a post-freeze addition.

In [ ]:
freeze = read_json('results/metrics/reproducibility_freeze_manifest.json')
try:
    current_commit = subprocess.run(
        ['git', 'rev-parse', 'HEAD'], cwd=ROOT, check=True, capture_output=True, text=True
    ).stdout.strip()
    freeze_tag_commit = subprocess.run(
        ['git', 'rev-list', '-n', '1', 'experiment-v1.0'],
        cwd=ROOT, check=True, capture_output=True, text=True
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    current_commit = 'unavailable outside a Git checkout'
    freeze_tag_commit = 'unavailable outside a Git checkout'

freeze_status = pd.DataFrame([
    ('Current repository commit', current_commit),
    ('Experiment tag', 'experiment-v1.0'),
    ('Experiment tag commit', freeze_tag_commit),
    ('Frozen source commit', freeze['frozen_source_commit']),
    ('Freeze version', freeze['freeze_version']),
    ('Dataset SHA-256', freeze['dataset']['sha256']),
    ('Python version at freeze', freeze['environment']['python_version']),
    ('Requirements SHA-256', freeze['environment']['requirements_sha256']),
    ('Scientific configuration frozen', freeze['validation']['scientific_configuration_frozen']),
], columns=['Item', 'Value'])
display(freeze_status)

## 3. Dataset Provenance and Dataset Integrity

The documented path is Physical Room → Sensor/Camera → ESP32/Raspberry Pi → Gateway → Azure/Cloud → Historical Storage → CSV Export → UTC Normalization → Validation → Canonical Twin State. Naive exported timestamps are interpreted as UTC without changing their clock values.

**Raw records and one-minute modeling samples are different data scopes.** The 2,027,520 raw records are not treated as 2,027,520 forecasting samples. All values below are loaded from official Stage-2 artifacts.

In [ ]:
dataset_summary = read_csv('results/tables/dataset_summary.csv')
data_quality = read_csv('results/tables/data_quality_summary.csv')
descriptive = read_csv('results/tables/descriptive_statistics.csv')
duplicates = read_csv('results/tables/duplicate_summary.csv')
temporal_gaps = read_csv('results/tables/temporal_gaps.csv')

summary_metrics = [
    'jumlah_record', 'jumlah_kolom', 'waktu_awal_utc', 'waktu_akhir_utc',
    'durasi_observasi_hari', 'jumlah_device', 'sha256', 'jumlah_gap'
]
quality_metrics = [
    'missing_cells', 'exact_duplicate_rows', 'duplicate_timestamp_rows',
    'timestamp_parse_failures', 'temporal_gaps'
]
display(dataset_summary[dataset_summary['metrik'].isin(summary_metrics)].reset_index(drop=True))
display(data_quality[data_quality['metrik'].isin(quality_metrics)].reset_index(drop=True))
display(duplicates)
display(descriptive[['variable', 'unit', 'count', 'missing', 'mean', 'sample_std', 'min', 'median', 'max']])
display(pd.DataFrame({
    'Temporal-gap statistic': ['Gap count', 'Minimum gap (s)', 'Median gap (s)', 'Maximum gap (s)'],
    'Value': [len(temporal_gaps), temporal_gaps['gap_seconds'].min(), temporal_gaps['gap_seconds'].median(), temporal_gaps['gap_seconds'].max()]
}))

### 3.1 Eight Original Raw Records

The table below contains the first eight data records from the original `sensor_data.csv`, not synthetic examples or one-minute aggregates. Original field names, timestamp strings, numeric text and file order are preserved. The initial high-power record and the irregular timestamp intervals have not been removed. This excerpt illustrates the schema only; it is not statistically representative of the complete dataset.

The source file was checked locally against the frozen dataset SHA-256, and the eight records were compared directly with that file. The small excerpt and its provenance manifest are stored separately under `docs/evidence/`; no full raw export or frozen result is replaced. In zero-upload reviewer mode, the checks below verify the excerpt checksum and its recorded source-hash link, **not an independent recheck of the unavailable full raw file**.

In [ ]:
raw_sample_manifest = read_json('docs/evidence/raw_dataset_sample_manifest.json')
raw_sample_path = ROOT / raw_sample_manifest['sample_file']
raw_sample = pd.read_csv(raw_sample_path, dtype=str, keep_default_na=False)
assert raw_sample_manifest['source_sha256'] == freeze['dataset']['sha256']
assert hashlib.sha256(raw_sample_path.read_bytes()).hexdigest() == raw_sample_manifest['sample_sha256']
assert list(raw_sample.columns) == raw_sample_manifest['source_columns']
assert len(raw_sample) == raw_sample_manifest['sample_record_count'] == 8
assert raw_sample_manifest['source_data_row_numbers'] == list(range(1, 9))
raw_sample.index = pd.Index(raw_sample_manifest['source_data_row_numbers'], name='Source data row (1-based)')
with pd.option_context('display.max_columns', None):
    display(raw_sample)
display(pd.DataFrame([
    ('Full source SHA-256', raw_sample_manifest['source_sha256']),
    ('Selection policy', raw_sample_manifest['selection_policy']),
    ('Local source verification date', raw_sample_manifest['verification_date']),
    ('Excerpt checksum and recorded source-hash link', 'PASS'),
    ('Full-source-to-excerpt comparison in this Colab run', 'Not repeated; full raw file is not required'),
], columns=['Raw-sample provenance', 'Value']))

### 3.2 Occupancy Acquisition: Documented Mechanism and Unresolved Measurement Evidence

The research provenance describes camera-derived occupancy from a Raspberry Pi and a latest-available-snapshot gateway path. The source audit records both YOLOv3-tiny and YOLOv8 variants, so **the exact detector deployed during collection is not established by the CSV**. A separately inspected, commit-identified YOLOv3-tiny implementation documents person-box counting, a face-count fallback, per-successful-frame processing and a nominal publication rule. These implementation facts must not be presented as a verified deployment history for the evaluated dataset.

In particular, a requested camera FPS, a nominal publication interval, combined CSV-record spacing and the one-minute modeling grid are different quantities. The dataset does not supply independent camera timestamps or an annotated counting-accuracy study. Realized occupancy update cadence, counting uncertainty and snapshot staleness remain unavailable; detector score thresholds are not counting-accuracy percentages.

In [ ]:
acquisition_evidence = read_json('docs/evidence/occupancy_acquisition_evidence.json')
assert acquisition_evidence['dataset_deployment_link_verified'] is False
acquisition_table = pd.DataFrame(acquisition_evidence['items']).rename(columns={
    'item': 'Item', 'documented_detail': 'Documented detail',
    'evidence_status': 'Evidence status', 'source': 'Source',
})
display_readable_table(acquisition_table)
inspected = acquisition_evidence['inspected_implementation']
implementation_url = f"{inspected['repository']}/blob/{inspected['commit']}/{inspected['file']}"
display(Markdown(
    f"**Inspected implementation only:** [source at commit `{inspected['commit'][:12]}`]({implementation_url})  \n"
    f"**File SHA-256:** `{inspected['sha256']}`  \n"
    '**Deployment during dataset collection:** not verified. The source file is not executed here.'
))
display(Markdown('**Additional evidence still needed:**\n\n' + '\n'.join(
    f'- {item}' for item in acquisition_evidence['additional_evidence_needed']
)))

## 4. Canonical Twin State

Each source record is represented through `timestamp_utc`, `room_id`, `device_id`, `environment`, `electrical`, `occupancy`, `data_quality`, and `provenance`. Source-to-canonical mappings preserve the documented units and values. `room_id` remains `unresolved`, and `staleness_seconds` remains unavailable because an independent camera timestamp is not present.

For this evaluated dataset and configuration, all 2,027,520 transformations succeeded, schema conformity and required telemetry completeness were 100%, and value-preservation mismatches were zero. Deterministic ordering was verified, while **204 records**—not the entire dataset—were re-transformed for determinism rechecks. These findings do not establish a universal ontology, a universal standard, perfect physical-digital synchronization, measured latency, or occupancy-staleness performance.

In [ ]:
canonical_mapping = read_csv('results/tables/canonical_field_mapping.csv')
canonical_evaluation = read_csv('results/tables/canonical_state_evaluation.csv')
canonical_quality = read_csv('results/tables/digital_twin_data_quality.csv')
canonical_metrics = [
    'total_raw_records_evaluated', 'successful_canonical_transformations',
    'schema_conformity_rate', 'required_field_completeness_rate',
    'value_preservation_mismatches', 'deterministic_ordering_verified',
    'records_rechecked', 'determinism_failures'
]
display(canonical_mapping[['source_field', 'canonical_field', 'unit', 'transformation', 'mapping_status']])
display(canonical_evaluation[canonical_evaluation['metric'].isin(canonical_metrics)].reset_index(drop=True))
display(canonical_quality[canonical_quality['item'].isin(['room_id_resolved', 'staleness_seconds_available', 'occupancy_staleness_calculable'])])

## 5. One-Minute Modeling Dataset

The temporal grid contains 128,289 one-minute bins. Empty bins remain empty: no forward filling and no interpolation are applied. Outliers are not silently deleted. Eligibility exclusions protect feature history, the exact future target, temporal-gap integrity, and split boundaries; they are methodological safeguards rather than arbitrary record deletion.

In [ ]:
modeling_summary = read_csv('results/tables/modeling_dataset_summary.csv')
exclusions = read_csv('results/tables/sample_exclusion_summary.csv')
preprocessing = read_csv('results/tables/preprocessing_report.csv')
display(modeling_summary)
display(exclusions)
display(preprocessing[['urutan', 'transformasi', 'record_sebelum', 'record_setelah', 'jumlah_terdampak', 'tindakan']])

### 5.1 Exact One-Minute Aggregation per Variable

The executed implementation uses the arithmetic **mean** of finite observations for temperature, humidity, voltage, current and power, and the **last valid integer occupancy observation** within each UTC minute bin. Median aggregation is not used. A bin is indexed by the timestamp rounded down to the minute and covers `[t, t + 1 min)`. Source rows must be chronological; the last valid occupancy is selected in that source order. Missing or non-finite values are not converted to zero; if no valid value exists for a variable, its aggregate is missing.

Occupancy zero is a valid observation. Reindexing retains empty minute bins, with no forward filling or interpolation. The one-minute grid is a modeling cadence, not evidence of the camera's update cadence. The code displayed below is read directly from the existing source functions; it is not a replacement aggregation algorithm.

In [ ]:
from src.features.time_series import MinuteAccumulator
from src.twin_state.canonical import RAW_COLUMNS

aggregation_variables = [
    ('temperature_c', '°C'), ('humidity_percent', '%'),
    ('voltage_v', 'V'), ('current_a', 'A'), ('power_w', 'W'),
    ('occupancy_count', 'persons'),
]
aggregation_table = pd.DataFrame([
    {
        'Raw field': RAW_COLUMNS[name], 'Modeling variable': name, 'Unit': unit,
        'Within-minute function': 'Last valid integer observation' if name == 'occupancy_count' else 'Arithmetic mean of finite observations',
        'No valid observation': 'Missing (NaN), not zero',
        'Implementation': 'src/features/time_series.py: _raw_values; MinuteAccumulator.add/finish; resample_sensor_csv',
    } for name, unit in aggregation_variables
])
display_readable_table(aggregation_table)
display(Code(inspect.getsource(MinuteAccumulator.add) + '\n' + inspect.getsource(MinuteAccumulator.finish), language='python'))

## 6. Forecast Target and Leakage Controls

**Target:** `power(t + 30 min)`  
**Unit:** W

Features are backward-looking or available at time `t`; the target is matched to exactly +30 minutes. Horizons crossing gaps and targets crossing split boundaries are excluded. Scaling is fitted on training data only, and test data are not used for hyperparameter selection. The protocol uses no random split, cross-validation, or rolling-origin validation.

In [ ]:
feature_definition = read_csv('results/tables/feature_definition.csv')
assert feature_definition['source_offset_max'].max() <= 0
leakage_controls = pd.DataFrame([
    ('Maximum feature source offset', f"{feature_definition['source_offset_max'].max()} min", 'PASS'),
    ('Exact target horizon', '30 min', 'PASS'),
    ('Gap-crossing horizons excluded', int(exclusions.loc[exclusions['reason'].eq('forecast_horizon_melintasi_gap'), 'sample_count'].iloc[0]), 'PASS'),
    ('Split-boundary targets excluded', int(exclusions.loc[exclusions['reason'].eq('target_melintasi_split_boundary'), 'sample_count'].iloc[0]), 'PASS'),
    ('Scaler fit split', 'train', 'PASS'),
    ('Hyperparameter selection set', 'validation', 'PASS'),
], columns=['Control', 'Observed', 'Status'])
display(leakage_controls)
display(feature_definition[['feature', 'group', 'source_offset_min', 'source_offset_max', 'unit']])

## 7. Chronological Data Split

The temporal-grid policy is chronological **70% train / 15% validation / 15% test**. After eligibility filtering, the usable counts are 85,242 train, 18,330 validation, and 18,126 test samples. The slight differences from exact percentages arise from eligibility filtering and boundary protection.

In [ ]:
temporal_split = read_csv('results/tables/temporal_split.csv')
display(temporal_split)
assert temporal_split.set_index('split')['usable_samples'].to_dict() == {
    'train': 85242, 'validation': 18330, 'test': 18126
}

## 8. Baseline Power Forecasting

Stage 3 compares (1) Persistence Baseline, (2) Historical-Power Ridge, and (3) Non-Occupancy Multivariate Ridge. The reported metrics are MAE, RMSE, and R² on validation and held-out test data. Non-Occupancy Multivariate Ridge was selected for the Stage-3 diagnostic figures using validation MAE—not test performance.

In [ ]:
baseline = read_csv('results/tables/baseline_model_comparison.csv')
baseline_columns = [
    'model', 'selected_alpha', 'validation_mae_w', 'validation_rmse_w', 'validation_r2',
    'test_mae_w', 'test_rmse_w', 'test_r2', 'selected_from_validation_for_figures'
]
display(baseline[baseline_columns])

### 8.1 Independent Metric Recomputation from Frozen Predictions

This reviewer-facing check independently recomputes MAE, RMSE, and R² directly from the committed held-out prediction CSV files. It does not fit or retrain any model. Stage-3 prediction columns are discovered from the CSV schema and the validation-selected model recorded in the official comparison table. Occupancy-ablation metrics are recomputed for both the paired control and treatment predictions.

In [ ]:
METRIC_TOLERANCE = 1e-9

def regression_metrics(actual, predicted):
    actual_values = np.asarray(actual, dtype=float)
    predicted_values = np.asarray(predicted, dtype=float)
    residual = predicted_values - actual_values
    mae = float(np.mean(np.abs(residual)))
    rmse = float(np.sqrt(np.mean(np.square(residual))))
    r2 = float(1.0 - np.sum(np.square(residual)) / np.sum(np.square(actual_values - actual_values.mean())))
    return {'MAE': mae, 'RMSE': rmse, 'R²': r2}

def metric_audit_rows(model_label, official_values, recomputed_values):
    rows = []
    for metric in ('MAE', 'RMSE', 'R²'):
        difference = abs(float(official_values[metric]) - float(recomputed_values[metric]))
        rows.append((model_label, metric, official_values[metric], recomputed_values[metric], difference, 'PASS' if difference <= METRIC_TOLERANCE else 'FAIL'))
    return rows

stage3_predictions_audit = read_csv('results/metrics/test_predictions.csv')
stage3_selected = baseline.loc[baseline['selected_from_validation_for_figures'].astype(str).str.lower().eq('true')].iloc[0]
stage3_actual_column = next(column for column in stage3_predictions_audit.columns if column == 'actual_power_w')
stage3_prediction_columns = [column for column in stage3_predictions_audit.columns if column.endswith('_prediction_w')]
stage3_selected_prediction_column = f"{stage3_selected['model']}_prediction_w"
assert stage3_selected_prediction_column in stage3_prediction_columns
stage3_recomputed = regression_metrics(stage3_predictions_audit[stage3_actual_column], stage3_predictions_audit[stage3_selected_prediction_column])
stage3_official = {'MAE': stage3_selected['test_mae_w'], 'RMSE': stage3_selected['test_rmse_w'], 'R²': stage3_selected['test_r2']}
stage3_metric_audit = pd.DataFrame(
    metric_audit_rows(str(stage3_selected['model']), stage3_official, stage3_recomputed),
    columns=['Model', 'Metric', 'Official Value', 'Recomputed Value', 'Absolute Difference', 'Status'],
)
display(Markdown(f'**Stage 3 actual column:** `{stage3_actual_column}`  \n**Selected prediction column:** `{stage3_selected_prediction_column}`'))
display(stage3_metric_audit)
assert len(stage3_predictions_audit) == 18126 and stage3_metric_audit['Status'].eq('PASS').all()

ablation_predictions_audit = read_csv('results/metrics/occupancy_ablation_test_predictions.csv')
ablation_official = read_csv('results/tables/occupancy_ablation_comparison.csv').set_index('role')
ablation_rows = []
ablation_recomputed = {}
for role, prediction_column in [('control', 'control_prediction_w'), ('treatment', 'treatment_prediction_w')]:
    recomputed = regression_metrics(ablation_predictions_audit['actual_power_w'], ablation_predictions_audit[prediction_column])
    ablation_recomputed[role] = recomputed
    official = {
        'MAE': ablation_official.loc[role, 'test_mae_w'],
        'RMSE': ablation_official.loc[role, 'test_rmse_w'],
        'R²': ablation_official.loc[role, 'test_r2'],
    }
    ablation_rows.extend(metric_audit_rows(role, official, recomputed))
ablation_metric_audit = pd.DataFrame(
    ablation_rows, columns=['Model', 'Metric', 'Official Value', 'Recomputed Value', 'Absolute Difference', 'Status']
)
control_mae = ablation_recomputed['control']['MAE']
treatment_mae = ablation_recomputed['treatment']['MAE']
recomputed_improvement_w = control_mae - treatment_mae
recomputed_improvement_percent = recomputed_improvement_w / control_mae * 100.0
official_treatment = ablation_official.loc['treatment']
improvement_audit = pd.DataFrame([
    ('MAE improvement (W)', official_treatment['mae_improvement_w_vs_control'], recomputed_improvement_w),
    ('MAE improvement (%)', official_treatment['mae_percentage_improvement'], recomputed_improvement_percent),
], columns=['Metric', 'Official Value', 'Recomputed Value'])
improvement_audit['Absolute Difference'] = (improvement_audit['Official Value'] - improvement_audit['Recomputed Value']).abs()
improvement_audit['Status'] = np.where(improvement_audit['Absolute Difference'] <= METRIC_TOLERANCE, 'PASS', 'FAIL')
display(ablation_metric_audit)
display(improvement_audit)
assert len(ablation_predictions_audit) == 18126
assert ablation_metric_audit['Status'].eq('PASS').all() and improvement_audit['Status'].eq('PASS').all()

## 9. Full Held-Out Test Forecast Figures

These post-freeze manuscript figures use all existing frozen prediction rows. They do not alter, smooth, or retrain the scientific outputs. The Stage-3 panel uses the validation-selected Non-Occupancy Multivariate Ridge prediction; the Stage-4 panel uses the occupancy-aware treatment prediction.

In [ ]:
stage3_predictions = read_csv('results/metrics/test_predictions.csv')
stage4_predictions = read_csv('results/metrics/occupancy_ablation_test_predictions.csv')
figure_scope = pd.DataFrame([
    ('Stage 3 complete held-out forecast', len(stage3_predictions), 'Non-Occupancy Multivariate Ridge'),
    ('Stage 4 complete held-out forecast', len(stage4_predictions), 'Occupancy-Aware Multivariate Ridge'),
], columns=['Figure', 'Held-out samples', 'Prediction series'])
assert len(stage3_predictions) == len(stage4_predictions) == 18126
display(figure_scope)
show_figure('paper/figures/forecast_actual_vs_predicted_full_test.png')
show_figure('paper/figures/occupancy_aware_actual_vs_predicted_full_test.png')

## 10. Occupancy Ablation

**Control:** Non-Occupancy Multivariate Ridge  
**Treatment:** Occupancy-Aware Multivariate Ridge

The paired comparison uses the same timestamps, target, split, preprocessing and sample mask, Ridge model family, alpha candidates, and tuning metric. The following checks are read directly from the official fairness artifact.

In [ ]:
fair_ablation = read_csv('results/tables/fair_ablation_check.csv')
display(fair_ablation)
assert fair_ablation['passed'].astype(str).str.lower().eq('true').all()

### 10.1 Exact Definitions of the Eleven Occupancy-Derived Features

Let `O_t` denote the last valid occupancy value in minute bin `t`. Treatment adds exactly **11** features to the 20-feature control: current `O_t`; four lags at **1, 5, 15 and 30 minutes**; and rolling **mean and maximum** for windows of **5, 15 and 30 minutes**. A lag is `O_(t-k)`. A rolling window of length `W` includes exactly `t-W+1, ..., t`, including the current minute bin and excluding future bins.

The source uses `rolling(window=W, min_periods=W)`: all `W` minute values must be valid, otherwise that rolling feature is missing. The one-minute grid makes row shifts equal minute shifts; a lag is not a shift in irregular raw-record order. Incomplete feature windows are handled by the frozen common-sample eligibility policy, not by imputing occupancy. The table is read from the official Stage-4 feature artifact and cross-checked against both source definitions and the configured treatment-minus-control list.

In [ ]:
import yaml
from src.features.time_series import OCCUPANCY_FEATURES, FEATURE_DEFINITIONS, add_forecasting_features

ablation_feature_definition = read_csv('results/tables/occupancy_ablation_feature_definition.csv')
occupancy_definitions = ablation_feature_definition.loc[
    ablation_feature_definition['occupancy_feature'].astype(str).str.lower().eq('true')
].copy().reset_index(drop=True)
with (ROOT / 'configs/features.yaml').open(encoding='utf-8') as handle:
    feature_config = yaml.safe_load(handle)
feature_sets = feature_config['model_feature_sets']
control_features = feature_sets['non_occupancy_multivariate_ridge']
treatment_features = feature_sets['occupancy_multivariate_ridge']
configured_occupancy_features = [name for name in treatment_features if name not in control_features]
assert list(occupancy_definitions['feature']) == OCCUPANCY_FEATURES == configured_occupancy_features
assert len(occupancy_definitions) == 11 and len(control_features) == 20 and len(treatment_features) == 31
for _, row in occupancy_definitions.iterrows():
    definition = FEATURE_DEFINITIONS[row['feature']]
    for field in ('description', 'source_offset_min', 'source_offset_max', 'unit'):
        assert row[field] == definition[field]
assert occupancy_definitions['source_offset_max'].max() == 0
display_readable_table(occupancy_definitions[[
    'feature', 'description', 'source_offset_min', 'source_offset_max', 'unit',
    'used_by_control', 'used_by_treatment',
]])
feature_source = inspect.getsource(add_forecasting_features)
occupancy_source = feature_source[
    feature_source.index('    for lag in OCCUPANCY_LAGS_MINUTES:'):
    feature_source.index('    minute_of_day =')
]
display(Code(occupancy_source, language='python'))
print('All 11 occupancy definitions match frozen artifact, source and configuration: PASS')

## 11. Occupancy Ablation Results

The occupancy-aware treatment has a modestly lower point-estimate test error than the control. Its MAE point-estimate improvement is approximately 0.99%.

> Occupancy produced a modest point-estimate improvement, but the moving-block bootstrap confidence interval included zero; the experiment therefore does not establish a consistent predictive improvement.

This is neither a causal occupancy effect nor proof of improvement.

In [ ]:
ablation = read_csv('results/tables/occupancy_ablation_comparison.csv')
bootstrap = read_csv('results/metrics/occupancy_ablation_bootstrap.csv')
display(ablation[[
    'model', 'role', 'selected_alpha', 'test_mae_w', 'test_rmse_w', 'test_r2',
    'mae_improvement_w_vs_control', 'mae_percentage_improvement'
]])
display(bootstrap)
assert float(bootstrap['mae_improvement_ci_lower_w'].iloc[0]) <= 0 <= float(bootstrap['mae_improvement_ci_upper_w'].iloc[0])

## 12. Occupancy-Level Analysis

The official table covers every held-out test observation and reports the unequal sample count at each occupancy level. Comparisons should therefore not be interpreted as equally sized strata.

In [ ]:
occupancy_level = read_csv('results/tables/occupancy_level_error.csv')
display(occupancy_level)
assert int(occupancy_level['sample_count'].sum()) == 18126
show_figure('paper/figures/occupancy_level_error.png', width=950)

## 13. Decision-Support Inputs

Stage 6 uses the official Stage-4 occupancy-aware test predictions joined one-to-one with the one-minute modeling state. This is the repository implementation path; it should not be described as passing a serialized Canonical Twin State object directly into the rule engine.

In [ ]:
decision_manifest = read_json('results/metrics/decision_support_manifest.json')
alignment = decision_manifest['input_alignment']
display(pd.DataFrame([
    ('Source scope', decision_manifest['source_scope']),
    ('Forecast model retrained', decision_manifest['forecast_model_retrained']),
    ('Prediction rows', alignment['prediction_rows']),
    ('Merged rows', alignment['merged_rows']),
    ('Unmatched state rows', alignment['unmatched_state_rows']),
    ('Occupancy mismatch rows', alignment['occupancy_mismatch_rows']),
], columns=['Item', 'Value']))

## 14. Decision-Support Scenario Evaluation

The thresholds are **declared research scenario thresholds**. They are not validated comfort standards, safety limits, or human-validated thresholds. The evaluation reports internal scenario behavior and does not establish operational effectiveness.

In [ ]:
decision_summary = decision_manifest['summary']
rule_summary = read_csv('results/tables/decision_support_rule_summary.csv')
display(pd.DataFrame([
    ('Evaluated samples', decision_summary['evaluated_sample_count']),
    ('Samples with active recommendation', decision_summary['active_recommendation_sample_count']),
    ('Recommendation coverage', decision_summary['recommendation_coverage']),
    ('No-action samples', decision_summary['no_action_count']),
    ('Invalid inputs', decision_summary['invalid_input_count']),
    ('Contradictory recommendation samples', decision_summary['contradictory_recommendation_count']),
    ('Determinism failures', decision_summary['determinism_failure_count']),
], columns=['Metric', 'Value']))
display(rule_summary[['rule_id', 'rule_name', 'threshold_basis', 'recommendation_count', 'sample_coverage']])

## 15. Decision-Support Consistency

The nine official checks measure traceability, consistency, deterministic behavior, and internal rule operation. They do not measure human acceptance, causal energy savings, real-world behavioral effectiveness, or autonomous control.

In [ ]:
decision_consistency = read_csv('results/tables/decision_support_consistency.csv')
display(decision_consistency)
assert len(decision_consistency) == 9
assert decision_consistency['status'].eq('PASS').all()

## 16. Manuscript-Ready Figures

The following publication-oriented copies are stored under `paper/figures/`. They were produced after the freeze exclusively from frozen machine-readable predictions, tables, and manifests; historical figures in `results/figures/` remain untouched.

In [ ]:
manuscript_figures = [
    ('Stage 3 complete held-out forecast', 'paper/figures/forecast_actual_vs_predicted_full_test.png', '18,126 held-out test samples'),
    ('Stage 4 occupancy-aware complete held-out forecast', 'paper/figures/occupancy_aware_actual_vs_predicted_full_test.png', '18,126 held-out test samples'),
    ('Stage 3 forecast-error distribution', 'paper/figures/forecast_error_distribution_full_test.png', '18,126 held-out test residuals'),
    ('Occupancy ablation comparison', 'paper/figures/occupancy_ablation_comparison.png', 'Paired control/treatment metrics on 18,126 samples'),
    ('Occupancy-level error', 'paper/figures/occupancy_level_error.png', 'All 18,126 test samples grouped by occupancy'),
    ('Decision-support rule frequency', 'paper/figures/decision_support_rule_frequency.png', 'All five official rule categories'),
    ('Decision-support occupancy distribution', 'paper/figures/decision_support_occupancy_distribution.png', 'All 18,126 evaluated decision samples'),
    ('Decision-support forecast delta', 'paper/figures/decision_support_forecast_delta.png', 'All 18,126 evaluated decision samples grouped by emitted rule'),
]
figure_inventory = pd.DataFrame(manuscript_figures, columns=['Figure', 'Artifact', 'Data scope'])
figure_inventory['Exists'] = figure_inventory['Artifact'].map(lambda path: (ROOT / path).is_file())
display(figure_inventory)
for _, row in figure_inventory.iterrows():
    display(Markdown(f"### {row['Figure']}"))
    show_figure(row['Artifact'], width=900)

## 17. Figure Quality Rules

All manuscript PNG figures use English titles and labels, include units where applicable, and are exported at 300 DPI. Complete-test figures use all eligible held-out samples without smoothing or arbitrary temporal subsetting. Spikes and outliers remain visible. The historical seven-day diagnostic figure remains unchanged under `results/figures/` and is not presented as a complete-period figure here.

## 18. Decision-Support Configuration and Representative Outputs

This executable evidence section parses the frozen Stage-6 configuration into a compact rule table and selects representative traces directly from the official held-out scenario artifact. Threshold-based rules are declared research scenario thresholds, not validated safety or comfort standards.

In [ ]:
import yaml

with (ROOT / 'configs/decision_support.yaml').open(encoding='utf-8') as handle:
    decision_config = yaml.safe_load(handle)

def comparator_symbol(value):
    return {'greater_than': '>', 'greater_than_or_equal': '≥', 'equals': '=='}.get(value, value)

def trigger_condition(rule):
    if rule['threshold_basis'] == 'input_validation_requirement':
        return 'Required decision input is missing or invalid'
    if 'power_threshold_w' in rule:
        return f"occupancy_count == {rule['occupancy_equals']} AND (current OR forecast power) {comparator_symbol(rule['comparator'])} {rule['power_threshold_w']} W"
    if 'forecast_delta_threshold_w' in rule:
        return f"forecast_delta_w {comparator_symbol(rule['comparator'])} {rule['forecast_delta_threshold_w']} W"
    if 'temperature_threshold_c' in rule:
        return (
            f"occupancy_count > {rule['occupancy_min_exclusive']} AND "
            f"(temperature_c {comparator_symbol(rule['comparator'])} {rule['temperature_threshold_c']} °C "
            f"OR humidity_percent {comparator_symbol(rule['comparator'])} {rule['humidity_threshold_percent']}%)"
        )
    return 'No higher-priority active rule was triggered'

threshold_labels = {
    'declared_research_scenario_threshold': 'Declared Research Scenario Threshold',
    'input_validation_requirement': 'Input Validation Requirement',
    'rule_fallback': 'Rule Fallback',
}
decision_rule_rows = []
for rule in decision_config['rules'].values():
    decision_rule_rows.append({
        'Rule ID': rule['rule_id'],
        'Rule Name': rule['name'],
        'Trigger Condition': trigger_condition(rule),
        'Severity': rule['severity'],
        'Priority': rule['priority'],
        'Action Class': rule['action_class'],
        'Threshold Basis': threshold_labels.get(rule['threshold_basis'], rule['threshold_basis']),
        'Recommendation': rule['recommendation_text'],
    })
decision_rule_configuration = pd.DataFrame(decision_rule_rows).sort_values('Rule ID').reset_index(drop=True)
display(decision_rule_configuration)

### Representative Decision-Support Traces

Representative traces are shown for interpretability only. All aggregate Decision-Support results are calculated over the complete 18,126-sample held-out test evaluation.

In [ ]:
decision_scenarios = read_csv('results/tables/decision_support_scenarios.csv')
scenario_counts = decision_scenarios.groupby('rule_id').size().to_dict()
rule_occurrence_status = decision_rule_configuration[['Rule ID', 'Rule Name']].copy()
rule_occurrence_status['Held-Out Occurrences'] = rule_occurrence_status['Rule ID'].map(scenario_counts).fillna(0).astype(int)
rule_occurrence_status['Evaluation Status'] = rule_occurrence_status['Held-Out Occurrences'].map(
    lambda count: f'{count:,} occurrence(s); representative trace shown below.'
    if count > 0 else 'Not triggered in the held-out test evaluation.'
)
display(rule_occurrence_status)

representative_columns = [
    'timestamp_utc', 'occupancy_count', 'current_power_w', 'forecast_power_30m_w',
    'forecast_delta_w', 'temperature_c', 'humidity_percent', 'rule_id', 'rule_name',
    'severity', 'action_class', 'reason', 'recommendation_text',
]
representative_traces = (
    decision_scenarios.sort_values(['rule_id', 'timestamp_utc'])
    .groupby('rule_id', as_index=False, sort=True)
    .head(1)[representative_columns]
    .reset_index(drop=True)
)
display(representative_traces)
assert len(decision_scenarios) == 18126
assert set(representative_traces['rule_id']) == {rule_id for rule_id, count in scenario_counts.items() if count > 0}

## 19. Final Evidence Table

This table maps each research component to its official evidence and claim boundary.

In [ ]:
evidence_table = pd.DataFrame([
    ('Dataset', 'dataset_summary.csv; data_quality_summary.csv', '2,027,520 raw records', 'Verified scope, period, integrity, and ranges', 'One room, one gateway, one observation period'),
    ('Original Raw Sample', 'docs/evidence/raw_dataset_sample.csv; raw_dataset_sample_manifest.json', 'First eight original records', 'Local direct comparison to the SHA-256-matched source; excerpt checksum checked here', 'Illustrative schema excerpt, not a representative statistical sample'),
    ('One-Minute Aggregation', 'src/features/time_series.py: MinuteAccumulator; resample_sensor_csv', 'Six measured variables', 'Mean for five continuous variables; last valid integer occupancy', 'One-minute modeling cadence is not camera update cadence'),
    ('Occupancy Feature Definitions', 'occupancy_ablation_feature_definition.csv; configs/features.yaml; src/features/time_series.py', 'Exactly 11 additional treatment features', 'Current occupancy, four lags and six backward-looking rolling features agree', 'No future observations; complete rolling windows required'),
    ('Occupancy Acquisition', 'docs/evidence/occupancy_acquisition_evidence.json; docs/data_provenance.md', 'Documented source variants, not a verified dataset deployment', 'Implementation mechanism and nominal publication rule documented separately', 'Deployment link, realized cadence and counting uncertainty remain unverified'),
    ('Canonical State', 'canonical_state_evaluation.csv; canonical_field_mapping.csv', '2,027,520 raw records', 'All configured transformations succeeded; zero preservation mismatches', 'No latency or occupancy-staleness measurement'),
    ('Forecasting', 'baseline_model_comparison.csv; test_predictions.csv', '18,126 eligible held-out samples', 'MAE, RMSE, and R² for three Stage-3 baselines', 'No cross-building generalization'),
    ('Occupancy Ablation', 'occupancy_ablation_comparison.csv; fair_ablation_check.csv', '18,126 paired held-out samples', 'Treatment has modestly lower point-estimate error', 'No causal interpretation'),
    ('Bootstrap', 'occupancy_ablation_bootstrap.csv', '1,000 moving-block bootstrap iterations', 'Confidence interval includes zero', 'Consistent predictive improvement not established'),
    ('Decision Support', 'decision_support_manifest.json; decision_support_consistency.csv', '18,126 evaluated scenario samples', 'Traceable deterministic outputs under declared thresholds', 'No human outcome or real-world effectiveness evaluation'),
], columns=['Research Component', 'Evidence Artifact', 'Data Scope', 'Primary Result', 'Claim Boundary'])
display(evidence_table)

## 20. Evidence-Supported Conclusions

- Canonical transformation successfully processed all 2,027,520 raw records under the configured schema.
- Held-out forecasting metrics use the complete eligible test set of 18,126 samples.
- The occupancy-aware treatment produced a modestly lower point-estimate error than the control.
- The moving-block bootstrap confidence interval includes zero, so consistent occupancy improvement is not established.
- Decision support produced traceable deterministic recommendations under declared research scenario thresholds.
- Decision-support effectiveness, human response, causal energy savings, and autonomous control were not evaluated.

## 21. Data Availability

The full raw `sensor_data.csv` file is intentionally not committed to Git. Its expected SHA-256 is recorded in the freeze manifest, and the experiments can be reproduced when the matching source dataset is supplied through a local path such as the `SENSOR_DATA_PATH` environment variable. An eight-record raw-schema excerpt is provided separately under `docs/evidence/` for the requested reviewer table; it does not replace the full source dataset or establish full-dataset redistribution permission.

**TODO: choose final Data Availability Statement before submission based on data ownership and redistribution permission.**

## 22. Reviewer-Risk Checklist

The executable check below scans narrative cells for unsupported affirmative claims and reports the required technical terminology. Boundary statements that explicitly say an outcome was not evaluated are retained because they prevent overclaiming.

In [ ]:
notebook_path = ROOT / 'notebooks/00_complete_research_workflow.ipynb'
notebook_document = json.loads(notebook_path.read_text(encoding='utf-8'))
narrative = '\n'.join(
    ''.join(cell.get('source', []))
    for cell in notebook_document['cells']
    if cell.get('cell_type') == 'markdown'
).lower()
risky_affirmative_patterns = {
    'Energy used as the forecast quantity': r'energy forecasting',
    'Claimed realized savings': r'(demonstrates|achieves|proves) energy savings?',
    'Claimed causal occupancy effect': r'(causal effect of occupancy|occupancy causes)',
    'Claimed significant occupancy improvement': r'(statistically significant|significant occupancy improvement)',
    'Claimed autonomous operation': r'autonomous control (is|was) (implemented|validated|demonstrated)',
    'Claimed universal canonical standard': r'(is|provides|defines) (a )?universal canonical standard',
    'Claimed exact synchronization': r'exact synchronization (is|was) (verified|measured|established)',
    'Claimed measured end-to-end latency': r'end-to-end latency (is|was) measured',
    'Claimed cross-building generalization': r'cross-building generalization (is|was) (verified|established|demonstrated)',
    'Unsupported percentage-error metric': r'\bmape\b',
    'Unsupported forecast comparison test': r'diebold[–-]mariano',
    'Unsupported validation protocol': r'(uses|used|applies|applied) rolling[–-]origin',
    'Unsupported split protocol': r'(uses|used|applies|applied) (a )?random split',
}
risk_results = pd.DataFrame([
    (label, not bool(re.search(pattern, narrative, flags=re.IGNORECASE)))
    for label, pattern in risky_affirmative_patterns.items()
], columns=['Reviewer-risk check', 'Pass'])
terminology = pd.DataFrame([
    ('Target', 'Power(t + 30 min), W'),
    ('Evaluation metrics', 'MAE, RMSE, R²'),
    ('Uncertainty', 'Moving-block bootstrap'),
    ('Split', 'Chronological 70/15/15'),
], columns=['Item', 'Required terminology'])
display(risk_results)
display(terminology)
assert risk_results['Pass'].all()

## 23. Notebook Engineering Quality

This notebook uses nbformat 4.x, reads repository-relative artifacts, keeps execution output out of version control, and contains no scientific algorithm implementation. `RECOMPUTE_EXPERIMENTS` is fixed to `False`; any future optional reproduction must call existing `src/` modules and write only to a separate `reproduction_output/` directory.

In [ ]:
cells = notebook_document['cells']
cell_ids = [cell.get('id') for cell in cells]
stored_outputs = sum(len(cell.get('outputs', [])) for cell in cells if cell.get('cell_type') == 'code')
non_null_counts = sum(
    cell.get('execution_count') is not None for cell in cells if cell.get('cell_type') == 'code'
)
serialized = notebook_path.read_text(encoding='utf-8')
home_path_token = '/' + 'Users' + '/'
credential_assignment = re.compile(
    r'(?i)(api[_-]?key|client[_-]?secret|password)\s*[=:]'
)
engineering_checks = pd.DataFrame([
    ('nbformat major version', notebook_document.get('nbformat') == 4),
    ('Unique non-empty cell IDs', len(cell_ids) == len(set(cell_ids)) and all(cell_ids)),
    ('Stored outputs equal zero', stored_outputs == 0),
    ('Non-null execution counts equal zero', non_null_counts == 0),
    ('No absolute local home path', home_path_token not in serialized),
    ('No credential assignments', credential_assignment.search(serialized) is None),
    ('Read-only experiment mode', RECOMPUTE_EXPERIMENTS is False),
    ('Dataset location is environment-controlled', 'SENSOR_DATA_PATH' in os.environ or str(SENSOR_DATA_PATH) == 'data/raw/sensor_data.csv'),
], columns=['Engineering check', 'Pass'])
display(engineering_checks)
assert engineering_checks['Pass'].all()